# T18 · Bảng đặc trưng đối tác của Việt Nam

Mỗi dòng là một đối tác. Cơ cấu HS tính trên **xuất khẩu + nhập khẩu năm 2024**. CAGR tính trên tổng hai chiều từ **2015 đến 2024**, qua 9 khoảng thời gian.

**Đặc trưng mô hình** gồm tỷ trọng từng phần HS và CAGR. Kim ngạch USD, mã và tên đối tác dùng để đối chiếu hoặc mô tả. Ngưỡng 1 tỷ USD giúp giảm ảnh hưởng của các đối tác có quy mô nhỏ. Notebook in độ nhạy theo ba ngưỡng để nhóm xem xét tác động của lựa chọn này.

**Nguồn** — `data/clean/partner_section_year.csv` và `data/clean/partner_year.csv`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

YEAR = 2024
BASE_YEAR = 2015
MIN_TRADE_USD = 1_000_000_000
FLOWS = ("Export", "Import")
SECTIONS = "I II III IV V VI VII VIII IX X XI XII XIII XIV XV XVI XVII XVIII XIX XX XXI".split()
EXCLUDED_TOTAL_CODES = {"WLD", "WORLD", "ALL", "TOT", "TOTAL", "XXX"}

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data/clean/partner_year.csv").is_file()
     and (p / "data/clean/partner_section_year.csv").is_file()), None
)
if root is None:
    raise FileNotFoundError("Đặt notebook trong repo có hai CSV nguồn ở data/clean/.")
clean = root / "data/clean"
partner = pd.read_csv(clean / "partner_year.csv", dtype={"partner_code": "string", "partner_iso3": "string"})
section = pd.read_csv(clean / "partner_section_year.csv", dtype={"partner_code": "string", "section_code": "string"})
keys = ["year", "trade_flow", "partner_code"]
for table, grain in [(partner, keys), (section, keys + ["section_code"])]:
    if table[grain + ["trade_usd"]].isna().any().any() or table.duplicated(grain).any():
        raise ValueError(f"Thiếu dữ liệu hoặc trùng khóa {grain}.")
    if not np.isfinite(table["trade_usd"]).all() or table["trade_usd"].lt(0).any():
        raise ValueError("Kim ngạch phải hữu hạn và không âm.")
if not section["section_code"].isin(SECTIONS).all():
    raise ValueError("Có phần HS nằm ngoài danh sách I–XXI.")
if BASE_YEAR >= YEAR or MIN_TRADE_USD < 0:
    raise ValueError("Kiểm tra lại năm gốc và ngưỡng kim ngạch.")
print(f"Nguồn: {len(partner):,} dòng đối tác, {len(section):,} dòng đối tác–phần HS.")

Nguồn: 6,776 dòng đối tác, 91,966 dòng đối tác–phần HS.


## 1. Bảng tra đối tác và kiểm tra ghép

Dùng quy tắc M4 dựa trên mã ISO3 ba chữ cái và loại mã tổng hợp. Mã S19 thuộc nhóm gộp Other Asia, nes nên bị loại. Kiểm tra cả quan hệ mã đối tác → ISO3/tên trên toàn nguồn và kiểm tra ISO3 → mã đối tác trong các năm phân tích. Hai mã lịch sử của Sudan (SDN) thuộc các giai đoạn khác nhau.

In [2]:
identity = ["partner_code", "partner_iso3", "partner_name"]
lookup = partner[identity].drop_duplicates()
if lookup[identity].isna().any().any() or lookup.duplicated("partner_code").any():
    raise ValueError("Mã đối tác có thông tin định danh thiếu hoặc mâu thuẫn.")
valid = lookup["partner_iso3"].str.fullmatch(r"[A-Z]{3}", na=False)
valid &= ~lookup["partner_iso3"].isin(EXCLUDED_TOTAL_CODES)
valid_lookup = lookup.loc[valid].copy()
active_codes = partner.loc[partner["year"].isin([BASE_YEAR, YEAR]) & partner["trade_flow"].isin(FLOWS), "partner_code"]
active_lookup = valid_lookup.loc[valid_lookup["partner_code"].isin(active_codes)]
if active_lookup.duplicated("partner_iso3").any():
    raise ValueError("Một ISO3 hợp lệ ứng với nhiều mã đối tác trong các năm phân tích.")
merged = section.merge(lookup, on="partner_code", how="left", validate="many_to_one", indicator=True)
if len(merged) != len(section) or merged["_merge"].ne("both").any():
    raise ValueError("Ghép sai số dòng hoặc còn mã đối tác chưa có trong bảng tra.")
valid_codes = valid_lookup["partner_code"]
print(f"Ghép: {len(section):,} → {len(merged):,} dòng, tất cả đều khớp.")
print("Mã bị loại:", ", ".join(lookup.loc[~valid, "partner_iso3"]))

Ghép: 91,966 → 91,966 dòng, tất cả đều khớp.
Mã bị loại: S19


## 2. Tỷ trọng HS và CAGR

Tỷ trọng phần HS (%) = kim ngạch của phần / tổng kim ngạch đối tác năm 2024 × 100. Ô phần HS vắng được gán 0 **sau khi** xác nhận tổng kim ngạch theo phần khớp với bảng đối tác ở từng chiều. Đây là 0 trong dữ liệu ghi nhận.

CAGR (%) = `((trade_2024 / trade_2015) ** (1/9) - 1) * 100`. Đối tác thiếu tổng kim ngạch năm gốc hoặc có giá trị gốc bằng 0 được loại khỏi bộ mô hình và được báo riêng. Kim ngạch năm gốc được cộng trên các chiều có bản ghi trong nguồn.

In [3]:
time_mask = partner["year"].isin([BASE_YEAR, YEAR]) & partner["trade_flow"].isin(FLOWS)
valid_partner = partner.loc[time_mask & partner["partner_code"].isin(valid_codes)]
current_sections = merged.loc[
    merged["year"].eq(YEAR) & merged["trade_flow"].isin(FLOWS) & merged["partner_code"].isin(valid_codes)
].copy()
partner_flow = valid_partner.loc[valid_partner["year"].eq(YEAR)].set_index(keys)["trade_usd"]
section_flow = current_sections.groupby(keys)["trade_usd"].sum()
check = pd.concat([partner_flow.rename("partner"), section_flow.rename("sections")], axis=1)
if check.isna().any().any() or not np.allclose(check["partner"], check["sections"], rtol=0, atol=.01):
    raise ValueError("Tổng phần HS lệch bảng đối tác theo năm/chiều. Kiểm tra trước khi gán 0.")
print(f"Đối soát năm {YEAR}: {len(check):,} khóa đối tác–chiều, lệch lớn nhất "
      f"{(check['partner']-check['sections']).abs().max():,.2f} USD.")

yearly = valid_partner.groupby(["partner_code","year"])["trade_usd"].sum().unstack("year")
totals = yearly[[BASE_YEAR,YEAR]].rename(columns={BASE_YEAR:"trade_base_usd",YEAR:"trade_analysis_usd"})
for threshold in sorted({500_000_000, MIN_TRADE_USD, 2_000_000_000}):
    hit = totals["trade_analysis_usd"].ge(threshold) & totals["trade_analysis_usd"].gt(0)
    print(f"Ngưỡng {threshold:,.0f} USD: {hit.sum()} đạt ngưỡng, {(hit & totals['trade_base_usd'].gt(0)).sum()} có CAGR.")
eligible = totals.loc[totals["trade_analysis_usd"].ge(MIN_TRADE_USD) & totals["trade_analysis_usd"].gt(0)]
invalid_growth = eligible.loc[~eligible["trade_base_usd"].gt(0)]
print("Đạt ngưỡng nhưng không tính được CAGR:",
      invalid_growth.join(valid_lookup.set_index("partner_code"))["partner_iso3"].tolist())
eligible = eligible.loc[eligible["trade_base_usd"].gt(0)].copy()
if len(eligible) < 2:
    raise ValueError("Cần ít nhất hai đối tác đủ dữ liệu để chuẩn hóa.")
eligible["cagr_pct"] = 100 * ((eligible["trade_analysis_usd"] / eligible["trade_base_usd"]) ** (1/(YEAR-BASE_YEAR)) - 1)

wide = current_sections.pivot_table(index="partner_code",columns="section_code",values="trade_usd",aggfunc="sum")
wide = wide.reindex(index=eligible.index, columns=SECTIONS).fillna(0)
shares = wide.div(eligible["trade_analysis_usd"], axis=0)*100
assert np.allclose(shares.sum(axis=1), 100, rtol=0, atol=1e-8)
shares.columns = [f"hs_{sec.lower()}_share_pct" for sec in SECTIONS]
raw = (valid_lookup.set_index("partner_code").join(eligible, how="inner").join(shares)
       .sort_values("partner_iso3").reset_index())
print(f"Bộ phân tích: {len(raw)} đối tác. Sai số tổng tỷ trọng tối đa: {(shares.sum(axis=1)-100).abs().max():.2e} điểm %.")

Đối soát năm 2024: 251 khóa đối tác–chiều, lệch lớn nhất 0.00 USD.
Ngưỡng 500,000,000 USD: 56 đạt ngưỡng, 56 có CAGR.
Ngưỡng 1,000,000,000 USD: 39 đạt ngưỡng, 39 có CAGR.
Ngưỡng 2,000,000,000 USD: 30 đạt ngưỡng, 30 có CAGR.
Đạt ngưỡng nhưng không tính được CAGR: []
Bộ phân tích: 39 đối tác. Sai số tổng tỷ trọng tối đa: 2.84e-14 điểm %.


## 3. Chuẩn hóa và xuất bộ dùng chung cho T19–T20

StandardScaler chạy trên các đặc trưng trong danh sách bàn giao theo đúng thứ tự. Cột hằng được giữ trong bảng gốc và loại khỏi mô hình. Các tỷ trọng HS có tổng bằng 100%, nên ma trận có phụ thuộc tuyến tính về cơ cấu. Notebook ghi số chiều và hạng ma trận để T19 diễn giải PCA. T19 và T20 dùng cùng bảng z-score và danh sách cột.

In [4]:
feature_candidates = shares.columns.tolist() + ["cagr_pct"]
matrix = raw[feature_candidates]
if not np.isfinite(matrix.to_numpy()).all():
    raise ValueError("Ma trận đặc trưng còn thiếu hoặc có giá trị vô hạn.")
constant = matrix.columns[matrix.nunique().le(1)].tolist()
feature_columns = [col for col in feature_candidates if col not in constant]
if not feature_columns:
    raise ValueError("Không còn đặc trưng có biến thiên.")

scaler = StandardScaler()
z = scaler.fit_transform(raw[feature_columns])
assert np.isfinite(z).all() and np.allclose(z.mean(axis=0), 0) and np.allclose(z.std(axis=0), 1)
standardized = pd.concat([raw[identity], pd.DataFrame(z, columns=feature_columns)], axis=1)
feature_list = pd.DataFrame({"feature_order": range(1,len(feature_columns)+1), "feature": feature_columns})
raw.to_csv(clean / "partner_features.csv", index=False)
standardized.to_csv(clean / "partner_features_standardized.csv", index=False)
feature_list.to_csv(clean / "partner_feature_columns.csv", index=False)

print("Cột hằng đã loại khỏi bảng chuẩn hóa:", constant or "không có")
print(f"Ma trận chuẩn hóa: {z.shape[0]} đối tác × {z.shape[1]} đặc trưng, hạng {np.linalg.matrix_rank(z)}.")
print(raw[["partner_iso3", "partner_name", "trade_analysis_usd", "cagr_pct"]].head().to_string(index=False))
print("Đã xuất partner_features.csv, partner_features_standardized.csv và partner_feature_columns.csv.")

Cột hằng đã loại khỏi bảng chuẩn hóa: không có
Ma trận chuẩn hóa: 39 đối tác × 22 đặc trưng, hạng 21.
partner_iso3 partner_name  trade_analysis_usd  cagr_pct
         ARG    Argentina        3.960657e+09  6.587858
         AUS    Australia        1.474646e+10 10.492548
         AUT      Austria        2.041734e+09  5.972845
         BEL      Belgium        1.650555e+09  3.277388
         BRA       Brazil        7.918902e+09  7.441032
Đã xuất partner_features.csv, partner_features_standardized.csv và partner_feature_columns.csv.
